# 0825_peace_016_type_expert_past_change_features


## 1. 설정, 경로 탐색과 실행 로그

`0825_peace_004_type_expert_walk_forward`와 동일한 분할·모델·평가 규칙을 유지하고, 입력 피처에만
"직전 동일 `inspection_type` 샘플 대비 센서 delta"에서 만든 집계 피처를 추가합니다.

이 노트북은 저장소 루트 또는 `notebooks/`에서 실행해도 같은 원본 파일과 로그 경로를 사용합니다.


In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_016_type_expert_past_change_features"
REFERENCE_EXPERIMENT_ID = "0825_peace_004_type_expert_walk_forward"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
MAD_SCALE_FACTOR = 1.4826
MIN_ROBUST_SCALE = 1e-6
DELTA_FEATURE_COLUMNS = [
    "delta_max_abs_z",
    "delta_mean_abs_z",
    "delta_p95_abs_z",
    "delta_abnormal_count_2",
    "delta_abnormal_count_3",
]

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}

REFERENCE_004_WALK_FORWARD = {
    "fixed_0.5": {
        "mean_pr_auc": 0.064566,
        "mean_recall": 0.083789,
        "min_recall": 0.039474,
        "recall_99_folds": 0,
        "mean_false_call_reduction": 0.997159,
    },
    "global_threshold": {
        "mean_pr_auc": 0.064566,
        "mean_recall": 0.969298,
        "min_recall": 0.907895,
        "recall_99_folds": 2,
        "mean_false_call_reduction": 0.169449,
    },
    "type_specific_thresholds": {
        "mean_pr_auc": 0.064566,
        "mean_recall": 0.923690,
        "min_recall": 0.789474,
        "recall_99_folds": 1,
        "mean_false_call_reduction": 0.333631,
    },
}
REFERENCE_004_VALIDATION = {
    "fixed_0.5": {
        "pr_auc": 0.4496564930298315,
        "precision": 0.45209580838323354,
        "recall": 0.42296918767507,
        "false_call_reduction": 0.995809384231377,
        "f1": 0.4370477568740955,
        "tp": 151,
        "fn": 206,
        "fp": 183,
        "tn": 43486,
    },
    "global_threshold": {
        "pr_auc": 0.4496564930298315,
        "precision": 0.02625917958608412,
        "recall": 0.9915966386554622,
        "false_call_reduction": 0.6993977421053836,
        "f1": 0.05116346292816881,
        "tp": 354,
        "fn": 3,
        "fp": 13127,
        "tn": 30542,
    },
    "type_specific_thresholds": {
        "pr_auc": 0.4496564930298315,
        "precision": 0.018324472203582305,
        "recall": 0.9943977591036415,
        "false_call_reduction": 0.5644965536192722,
        "f1": 0.03598580841358338,
        "tp": 355,
        "fn": 2,
        "fp": 19018,
        "tn": 24651,
    },
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info("reference_experiment=%s", REFERENCE_EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 16:20:45,177 | INFO | experiment=0825_peace_016_type_expert_past_change_features


2026-08-25 16:20:45,178 | INFO | reference_experiment=0825_peace_004_type_expert_walk_forward


2026-08-25 16:20:45,178 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 16:20:45,178 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 16:20:45,179 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 16:20:45,179 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 16:20:45,179 | INFO | log_file=docs/peace/0825_peace_016_type_expert_past_change_features.log


log saved to: docs/peace/0825_peace_016_type_expert_past_change_features.log


## 2. 원본 데이터, 매핑 검증과 causal delta 생성

첫 번째 익명 인덱스 열은 `record_id`로 이름만 바꾸며 원본 파일은 수정하지 않습니다.
모든 행은 `timestamp`, `record_id` 기준 stable sort 후 사용합니다.

새 피처는 각 센서값에서 "같은 `inspection_type`의 직전 샘플" 값을 뺀 delta입니다.
첫 관측은 비교 대상이 없으므로 0으로 둡니다.


In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
mapped_union = sorted(set().union(*(set(columns) for columns in feature_mapping.values())))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert set(mapped_union) <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

previous_timestamp = raw_df.groupby(TYPE_COLUMN, sort=False)[TIME_COLUMN].shift(1)
has_previous = previous_timestamp.notna()
assert (
    previous_timestamp.loc[has_previous].to_numpy()
    <= raw_df.loc[has_previous, TIME_COLUMN].to_numpy()
).all()

cumulative_rank = raw_df.groupby(TYPE_COLUMN, sort=False).cumcount()
delta_values = (
    raw_df.groupby(TYPE_COLUMN, sort=False)[mapped_union]
    .diff()
    .fillna(0.0)
    .astype(np.float32)
)
delta_column_map = {column: f"{column}__delta" for column in mapped_union}
delta_values = delta_values.rename(columns=delta_column_map)
raw_df = pd.concat([raw_df, delta_values], axis=1)
first_delta_columns = list(delta_column_map.values())
assert (
    raw_df.loc[cumulative_rank == 0, first_delta_columns].to_numpy(dtype=np.float32) == 0.0
).all()

delta_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "mapped_sensor_count": len(mapped_union),
        "delta_sensor_count": len(first_delta_columns),
        "causal_first_rows_zero": True,
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="delta_ready_data",
)
display(delta_summary)
logger.info(
    "data_verified rows=%d columns=%d mapped_sensors=%d delta_columns=%d",
    len(raw_df),
    raw_df.shape[1],
    len(mapped_union),
    len(first_delta_columns),
)


rows                                         440274
columns                                         143
inspection_types                                  5
mapped_sensor_count                              65
delta_sensor_count                               65
causal_first_rows_zero                         True
timestamp_start           1970-06-23 03:58:55+00:00
timestamp_end             1970-11-02 14:21:28+00:00
Name: delta_ready_data, dtype: object

2026-08-25 16:20:51,173 | INFO | data_verified rows=440274 columns=143 mapped_sensors=65 delta_columns=65


## 3. 타입별 유효 피처

각 전문가 모델은 공통 `meta_feat1~4`, `mapping.json`의 해당 타입 센서, 그리고 5개의 delta 집계 피처를 사용합니다.


In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
mapped_sensor_columns_by_type = {}
feature_columns_by_type = {}
delta_sensor_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    mapped_sensor_columns_by_type[inspection_type] = mapped_columns
    delta_sensor_columns_by_type[inspection_type] = [
        delta_column_map[column] for column in mapped_columns
    ]
    selected_columns = meta_columns + mapped_columns + DELTA_FEATURE_COLUMNS
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "delta_aggregate_features": len(DELTA_FEATURE_COLUMNS),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,delta_aggregate_features,total_model_features
inspection_type,,,,
0,4,44,5,53
1,4,52,5,61
2,4,65,5,74
3,4,65,5,74
4,4,21,5,30


2026-08-25 16:20:51,183 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'delta_aggregate_features': 5, 'total_model_features': 53}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'delta_aggregate_features': 5, 'total_model_features': 61}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'delta_aggregate_features': 5, 'total_model_features': 74}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'delta_aggregate_features': 5, 'total_model_features': 74}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'delta_aggregate_features': 5, 'total_model_features': 30}}


## 4. 시간순 Train/Validation/Test 분할

`004`와 동일하게 전체 행의 누적 비율에 가장 가까운 timestamp 그룹 끝을 경계로 사용합니다.

- 0~70%: Train
- 70~80%: Validation
- 80~100%: 최종 Test

이번 실험은 Test를 만들기만 하고 예측과 평가는 수행하지 않습니다.


In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask].copy()
validation_df = raw_df.loc[validation_mask].copy()
test_df = raw_df.loc[test_mask].copy()
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test_held_out_only",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "test_predictions_generated": False,
        "fixed_threshold_for_reference": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(split_summary)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy untouched=True threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test_held_out_only,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test        False
threshold_selected_on_test       False
test_predictions_generated       False
fixed_threshold_for_reference      0.5
Name: evaluation_policy, dtype: object

2026-08-25 16:20:51,696 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test_held_out_only', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 16:20:51,696 | INFO | test_policy untouched=True threshold=0.50


## 5. 평가 지표와 delta 집계 함수

PR-AUC, ROC-AUC, Accuracy, Precision, Recall, F1, TP/FN/FP/TN, False Call Reduction을 `004`와 동일하게 계산합니다.
Delta 집계 피처는 fold/final train에서만 median/MAD를 추정하고 이후 구간에는 그대로 적용합니다.


In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")


def fit_delta_reference(type_train: pd.DataFrame, inspection_type: int):
    delta_columns = delta_sensor_columns_by_type[inspection_type]
    delta_train = type_train[delta_columns]
    median = delta_train.median(axis=0)
    mad = delta_train.sub(median, axis=1).abs().median(axis=0) * MAD_SCALE_FACTOR
    scale = mad.mask(~np.isfinite(mad) | (mad < MIN_ROBUST_SCALE), 1.0).astype("float32")
    median = median.astype("float32")
    return {"delta_columns": delta_columns, "median": median, "scale": scale}


def build_delta_feature_frame(frame: pd.DataFrame, delta_reference: dict) -> pd.DataFrame:
    delta_values = frame[delta_reference["delta_columns"]].to_numpy(dtype=np.float32, copy=False)
    median = delta_reference["median"].to_numpy(dtype=np.float32, copy=False)
    scale = delta_reference["scale"].to_numpy(dtype=np.float32, copy=False)
    abs_z = np.abs((delta_values - median) / scale)
    features = pd.DataFrame(
        {
            "delta_max_abs_z": abs_z.max(axis=1).astype(np.float32),
            "delta_mean_abs_z": abs_z.mean(axis=1).astype(np.float32),
            "delta_p95_abs_z": np.quantile(abs_z, 0.95, axis=1).astype(np.float32),
            "delta_abnormal_count_2": (abs_z >= 2.0).sum(axis=1).astype(np.int16),
            "delta_abnormal_count_3": (abs_z >= 3.0).sum(axis=1).astype(np.int16),
        },
        index=frame.index,
    )
    assert np.isfinite(features.to_numpy(dtype=np.float32)).all()
    return features


def make_model_frame(frame: pd.DataFrame, inspection_type: int, delta_reference: dict) -> pd.DataFrame:
    base_columns = meta_columns + mapped_sensor_columns_by_type[inspection_type]
    delta_features = build_delta_feature_frame(frame, delta_reference)
    model_frame = pd.concat([frame[base_columns], delta_features], axis=1)
    assert model_frame.columns.tolist() == feature_columns_by_type[inspection_type]
    return model_frame


def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


2026-08-25 16:20:51,727 | INFO | threshold_selector_unit_test=PASS


## 6. 3-Fold Expanding Walk-forward 검증

첫 70% 개발 구간 안에서 Train을 누적 확장합니다. 각 Fold의 Calibration에서 임계값을 선택하고,
그 임계값을 바로 다음 미래 Evaluation에 고정 적용합니다.

| Fold | Train | Calibration | Evaluation |
|---|---:|---:|---:|
| Fold 1 | 0~30% | 30~40% | 40~50% |
| Fold 2 | 0~40% | 40~50% | 50~60% |
| Fold 3 | 0~50% | 50~60% | 60~70% |


In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end].copy(),
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ].copy(),
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ].copy(),
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 16:20:51,986 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

In [7]:
def fit_type_experts_for_fold(train_frame, calibration_frame, evaluation_frame, fold_name):
    calibration_probability = pd.Series(
        np.nan, index=calibration_frame.index, dtype="float64"
    )
    evaluation_probability = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    training_rows = []

    for inspection_type in inspection_types:
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type].copy()
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ].copy()
        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ].copy()
        y_train = type_train[TARGET].astype("int8")

        assert len(type_train) > 0
        assert len(type_calibration) > 0
        assert len(type_evaluation) > 0
        assert y_train.nunique() == 2
        assert type_calibration[TARGET].nunique() == 2

        delta_reference = fit_delta_reference(type_train, inspection_type)
        train_median_count = int((delta_reference["scale"] == 1.0).sum())
        logger.info(
            "walk_forward_fit_start fold=%s type=%d train_rows=%d train_positive=%d calibration_rows=%d calibration_positive=%d evaluation_rows=%d evaluation_positive=%d delta_scale_floor_count=%d",
            fold_name,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            len(type_calibration),
            int(type_calibration[TARGET].sum()),
            len(type_evaluation),
            int(type_evaluation[TARGET].sum()),
            train_median_count,
        )

        train_model_frame = make_model_frame(type_train, inspection_type, delta_reference)
        calibration_model_frame = make_model_frame(type_calibration, inspection_type, delta_reference)
        evaluation_model_frame = make_model_frame(type_evaluation, inspection_type, delta_reference)

        preprocessor = make_preprocessor(feature_columns_by_type[inspection_type])
        X_train = preprocessor.fit_transform(train_model_frame)
        X_calibration = preprocessor.transform(calibration_model_frame)
        X_evaluation = preprocessor.transform(evaluation_model_frame)
        assert np.isfinite(X_train.data if hasattr(X_train, "data") else X_train).all()
        assert np.isfinite(X_calibration.data if hasattr(X_calibration, "data") else X_calibration).all()
        assert np.isfinite(X_evaluation.data if hasattr(X_evaluation, "data") else X_evaluation).all()

        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, verbose=False)
        calibration_probability.loc[type_calibration.index] = model.predict_proba(
            X_calibration
        )[:, 1]
        evaluation_probability.loc[type_evaluation.index] = model.predict_proba(
            X_evaluation
        )[:, 1]

        training_rows.append(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "calibration_rows": len(type_calibration),
                "calibration_positive": int(type_calibration[TARGET].sum()),
                "evaluation_rows": len(type_evaluation),
                "evaluation_positive": int(type_evaluation[TARGET].sum()),
                "raw_features": len(feature_columns_by_type[inspection_type]),
                "encoded_features": X_train.shape[1],
            }
        )
        logger.info("walk_forward_fit_done fold=%s type=%d", fold_name, inspection_type)
        del preprocessor, model, X_train, X_calibration, X_evaluation
        gc.collect()

    assert calibration_probability.notna().all()
    assert evaluation_probability.notna().all()
    return calibration_probability, evaluation_probability, training_rows


walk_forward_threshold_rows = []
walk_forward_metric_rows = []
walk_forward_type_evaluation_rows = []
walk_forward_training_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    segments = walk_forward_segments[fold_name]
    calibration_frame = segments["calibration"]
    evaluation_frame = segments["evaluation"]
    calibration_probability, evaluation_probability, training_rows = (
        fit_type_experts_for_fold(
            segments["train"], calibration_frame, evaluation_frame, fold_name
        )
    )
    walk_forward_training_rows.extend(training_rows)

    global_selection = select_threshold(
        calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL
    )
    walk_forward_threshold_rows.append(
        {"fold": fold_name, "scope": "global", **global_selection}
    )

    thresholds_by_type_fold = {}
    type_evaluation_prediction = pd.Series(
        np.nan, index=evaluation_frame.index, dtype="float64"
    )
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[
            calibration_frame[TYPE_COLUMN] == inspection_type
        ]
        type_calibration_probability = calibration_probability.loc[
            type_calibration.index
        ]
        selection = select_threshold(
            type_calibration[TARGET],
            type_calibration_probability,
            min_recall=MIN_RECALL,
        )
        threshold = selection["threshold"]
        thresholds_by_type_fold[inspection_type] = threshold
        walk_forward_threshold_rows.append(
            {
                "fold": fold_name,
                "scope": f"type_{inspection_type}",
                **selection,
            }
        )

        type_evaluation = evaluation_frame.loc[
            evaluation_frame[TYPE_COLUMN] == inspection_type
        ]
        type_evaluation_probability = evaluation_probability.loc[type_evaluation.index]
        type_prediction = (type_evaluation_probability >= threshold).astype("int8")
        type_evaluation_prediction.loc[type_evaluation.index] = type_prediction
        type_metrics = evaluate_predictions(
            type_evaluation[TARGET], type_prediction, type_evaluation_probability
        )
        type_metrics.update(
            {
                "fold": fold_name,
                "inspection_type": inspection_type,
                "threshold": threshold,
            }
        )
        walk_forward_type_evaluation_rows.append(type_metrics)

    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(
            evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD
        ),
        "global_threshold": evaluate_probabilities(
            evaluation_frame[TARGET],
            evaluation_probability,
            global_selection["threshold"],
        ),
        "type_specific_thresholds": evaluate_predictions(
            evaluation_frame[TARGET],
            type_evaluation_prediction,
            evaluation_probability,
        ),
    }
    for strategy, metrics in strategy_metrics.items():
        walk_forward_metric_rows.append(
            {"fold": fold_name, "strategy": strategy, **metrics}
        )

    logger.info(
        "walk_forward_fold_done fold=%s global_threshold=%.8f metrics=%s",
        fold_name,
        global_selection["threshold"],
        strategy_metrics,
    )

walk_forward_threshold_summary = pd.DataFrame(walk_forward_threshold_rows).set_index(
    ["fold", "scope"]
)
walk_forward_evaluation_metrics = pd.DataFrame(walk_forward_metric_rows).set_index(
    ["fold", "strategy"]
)
walk_forward_type_evaluation = pd.DataFrame(
    walk_forward_type_evaluation_rows
).set_index(["fold", "inspection_type"])
walk_forward_training_summary = pd.DataFrame(walk_forward_training_rows).set_index(
    ["fold", "inspection_type"]
)

walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index()
    .groupby("strategy")
    .agg(
        folds=("fold", "nunique"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"),
        min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"),
        total_fn=("fn", "sum"),
    )
)
walk_forward_reference_comparison = (
    walk_forward_strategy_summary[
        [
            "mean_pr_auc",
            "mean_recall",
            "min_recall",
            "recall_99_folds",
            "mean_false_call_reduction",
        ]
    ]
    .join(pd.DataFrame.from_dict(REFERENCE_004_WALK_FORWARD, orient="index"), rsuffix="_ref004")
)
for metric in [
    "mean_pr_auc",
    "mean_recall",
    "min_recall",
    "recall_99_folds",
    "mean_false_call_reduction",
]:
    walk_forward_reference_comparison[f"{metric}_delta_vs_004"] = (
        walk_forward_reference_comparison[metric]
        - walk_forward_reference_comparison[f"{metric}_ref004"]
    )
logger.info(
    "walk_forward_strategy_summary=%s",
    walk_forward_strategy_summary.to_dict(orient="index"),
)
logger.info(
    "walk_forward_reference_comparison=%s",
    walk_forward_reference_comparison.to_dict(orient="index"),
)


2026-08-25 16:20:52,059 | INFO | walk_forward_fit_start fold=fold_1 type=0 train_rows=28277 train_positive=32 calibration_rows=8408 calibration_positive=11 evaluation_rows=6496 evaluation_positive=50 delta_scale_floor_count=34


2026-08-25 16:20:52,469 | INFO | walk_forward_fit_done fold=fold_1 type=0


2026-08-25 16:20:52,518 | INFO | walk_forward_fit_start fold=fold_1 type=1 train_rows=22698 train_positive=269 calibration_rows=3868 calibration_positive=20 evaluation_rows=2618 evaluation_positive=186 delta_scale_floor_count=42


2026-08-25 16:20:53,991 | INFO | walk_forward_fit_done fold=fold_1 type=1


2026-08-25 16:20:54,061 | INFO | walk_forward_fit_start fold=fold_1 type=2 train_rows=42288 train_positive=408 calibration_rows=16448 calibration_positive=92 evaluation_rows=18964 evaluation_positive=49 delta_scale_floor_count=59


...

2026-08-25 16:21:12,893 | INFO | walk_forward_fit_start fold=fold_3 type=4 train_rows=2771 train_positive=10 calibration_rows=698 calibration_positive=3 evaluation_rows=344 evaluation_positive=0 delta_scale_floor_count=11


2026-08-25 16:21:13,562 | INFO | walk_forward_fit_done fold=fold_3 type=4


2026-08-25 16:21:13,855 | INFO | walk_forward_fold_done fold=fold_3 global_threshold=0.00009221 metrics={'fixed_0.5': {'rows': 43853, 'positive_samples': 39, 'tn': 43785, 'fp': 29, 'fn': 37, 'tp': 2, 'accuracy': 0.9984949718377306, 'precision': 0.06451612903225806, 'recall': 0.05128205128205128, 'false_call_reduction': 0.9993381111060392, 'f1': 0.05714285714285714, 'roc_auc': 0.9202157605635947, 'pr_auc': 0.043625558887866525}, 'global_threshold': {'rows': 43853, 'positive_samples': 39, 'tn': 3910, 'fp': 39904, 'fn': 0, 'tp': 39, 'accuracy': 0.0900508517091191, 'precision': 0.0009763913576847007, 'recall': 1.0, 'false_call_reduction': 0.08924088190989181, 'f1': 0.0019508778950527738, 'roc_auc': 0.9202157605635947, 'pr_auc': 0.043625558887866525}, 'type_specific_thresholds': {'rows': 43853, 'positive_samples': 39, 'tn': 6328, 'fp': 37486, 'fn': 0, 'tp': 39, 'accuracy': 0.14518961074498893, 'precision': 0.001039307128580946, 'recall': 1.0, 'false_call_reduction': 0.14442872141324692, 'f1

2026-08-25 16:21:13,862 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06360323143609777, 'mean_recall': 0.07713868397594585, 'min_recall': 0.03289473684210526, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9966336597713658, 'min_false_call_reduction': 0.9951274191334584, 'total_tp': 55, 'total_fn': 462}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06360323143609777, 'mean_recall': 0.9758771929824562, 'min_recall': 0.9276315789473685, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.15434369663821232, 'min_false_call_reduction': 0.0026536121151118637, 'total_tp': 506, 'total_fn': 11}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06360323143609777, 'mean_recall': 0.9400629641588635, 'min_recall': 0.8355263157894737, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.27014943695905236, 'min_false_call_reduction': 0.14442872141324692, 'total_tp': 487, 'total_fn': 30}}


2026-08-25 16:21:13,863 | INFO | walk_forward_reference_comparison={'fixed_0.5': {'mean_pr_auc': 0.06360323143609777, 'mean_recall': 0.07713868397594585, 'min_recall': 0.03289473684210526, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9966336597713658, 'mean_pr_auc_ref004': 0.064566, 'mean_recall_ref004': 0.083789, 'min_recall_ref004': 0.039474, 'recall_99_folds_ref004': 0, 'mean_false_call_reduction_ref004': 0.997159, 'mean_pr_auc_delta_vs_004': -0.0009627685639022332, 'mean_recall_delta_vs_004': -0.006650316024054151, 'min_recall_delta_vs_004': -0.006579263157894741, 'recall_99_folds_delta_vs_004': 0, 'mean_false_call_reduction_delta_vs_004': -0.0005253402286342634}, 'global_threshold': {'mean_pr_auc': 0.06360323143609777, 'mean_recall': 0.9758771929824562, 'min_recall': 0.9276315789473685, 'recall_99_folds': 2, 'mean_false_call_reduction': 0.15434369663821232, 'mean_pr_auc_ref004': 0.064566, 'mean_recall_ref004': 0.969298, 'min_recall_ref004': 0.907895, 'recall_99_folds_ref00

## 7. Walk-forward 미래 Evaluation 결과

아래 지표는 각 Fold의 Calibration에서 임계값을 정한 뒤, 그 다음 미래 Evaluation에 고정 적용한 결과입니다.
`004` 기준값과의 차이도 함께 표시합니다.


In [8]:
display(
    walk_forward_threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]
    ]
)
display(
    walk_forward_evaluation_metrics[
        [
            "positive_samples",
            "pr_auc",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(
    walk_forward_type_evaluation[
        [
            "threshold",
            "positive_samples",
            "pr_auc",
            "recall",
            "false_call_reduction",
            "tp",
            "fn",
        ]
    ]
)
display(walk_forward_strategy_summary)
display(walk_forward_reference_comparison)
display(walk_forward_training_summary)


threshold  positive_samples    recall  false_call_reduction  \
fold   scope                                                                 
fold_1 global   0.000021               200  0.990000              0.008452   
       type_0   0.001250                11  1.000000              0.330952   
       type_1   0.001021                20  1.000000              0.308212   
       type_2   0.000019                92  1.000000              0.007459   
       type_3   0.000333                73  1.000000              0.156838   
       type_4   0.002461                 4  1.000000              0.000000   
fold_2 global   0.000505               326  0.990798              0.290937   
       type_0   0.000423                50  1.000000              0.372324   
       type_1   0.000505               186  0.994624              0.218339   
       type_2   0.000533                49  1.000000              0.318689   
       type_3   0.005937                39  1.000000              0.552571   
       type_4   0.003340                 2  1.000000              0.000000   
fold_3 global   0.000092               152  0.993421              0.130623   
       type_0   0.000059                14  1.000000              0.033441   
       type_1   0.000622                80  1.000000              0.332187   
       type_2   0.000092                32  1.000000              0.045047   
       type_3   0.000182                23  1.000000              0.447356   
       type_4   0.003684                 3  1.000000              0.000000   

                tp  fn  
fold   scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
fold   strategy                                                          
fold_1 fixed_0.5                              326  0.121657   0.183908   
       global_threshold                       326  0.121657   0.007422   
       type_specific_thresholds               326  0.121657   0.008535   
fold_2 fixed_0.5                              152  0.025527   0.024272   
       global_threshold                       152  0.025527   0.005066   
       type_specific_thresholds               152  0.025527   0.005961   
fold_3 fixed_0.5                               39  0.043626   0.064516   
       global_threshold                        39  0.043626   0.000976   
       type_specific_thresholds                39  0.043626   0.001039   

                                   recall  false_call_reduction        f1  \
fold   strategy                                                             
fold_1 fixed_0.5                 0.147239              0.995127  0.163543   
       global_threshold          1.000000              0.002654  0.014734   
       type_specific_thresholds  0.984663              0.146955  0.016923   
fold_2 fixed_0.5                 0.032895              0.995435  0.027933   
       global_threshold          0.927632              0.371137  0.010077   
       type_specific_thresholds  0.835526              0.519064  0.011838   
fold_3 fixed_0.5                 0.051282              0.999338  0.057143   
       global_threshold          1.000000              0.089241  0.001951   
       type_specific_thresholds  1.000000              0.144429  0.002076   

                                  tp   fn     fp     tn  
fold   strategy                                          
fold_1 fixed_0.5                  48  278    213  43501  
       global_threshold          326    0  43598    116  
       type_specific_thresholds  321    5  37290   6424  
fold_2 fixed_0.5                   5  147    201  43834  
       global_threshold          141   11  27692  16343  
       type_specific_thresholds  127   25  21178  22857  
fold_3 fixed_0.5                   2   37     29  43785  
       global_threshold           39    0  39904   3910  
       type_specific_thresholds   39    0  37486   6328

threshold  positive_samples    pr_auc    recall  \
fold   inspection_type                                                    
fold_1 0                 0.001250                50  0.021655  0.940000   
       1                 0.001021               186  0.211229  0.989247   
       2                 0.000019                49  0.307600  1.000000   
       3                 0.000333                39  0.465485  1.000000   
       4                 0.002461                 2  0.006154  1.000000   
fold_2 0                 0.000423                14  0.003828  0.714286   
       1                 0.000505                80  0.287762  1.000000   
       2                 0.000533                32  0.023067  0.781250   
       3                 0.005937                23  0.001631  0.391304   
       4                 0.003340                 3  0.004298  1.000000   
fold_3 0                 0.000059                 4  0.010404  1.000000   
       1                 0.000622                25  0.040050  1.000000   
       2                 0.000092                 7  0.024158  1.000000   
       3                 0.000182                 3  0.337634  1.000000   
       4                 0.003684                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
fold   inspection_type                                 
fold_1 0                            0.557400   47   3  
       1                            0.182155  184   2  
       2                            0.003436   49   0  
       3                            0.148929   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.576636   10   4  
       1                            0.443860   80   0  
       2                            0.207309   25   7  
       3                            0.660394    9  14  
       4                            0.000000    3   0  
fold_3 0                            0.038090    4   0  
       1                            0.165596   25   0  
       2                            0.045834    7   0  
       3                            0.351302    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.063603,0.077139,0.032895,0,0.996634,0.995127,55,462
global_threshold,3,0.063603,0.975877,0.927632,2,0.154344,0.002654,506,11
type_specific_thresholds,3,0.063603,0.940063,0.835526,1,0.270149,0.144429,487,30


,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,mean_pr_auc_ref004,mean_recall_ref004,min_recall_ref004,recall_99_folds_ref004,mean_false_call_reduction_ref004,mean_pr_auc_delta_vs_004,mean_recall_delta_vs_004,min_recall_delta_vs_004,recall_99_folds_delta_vs_004,mean_false_call_reduction_delta_vs_004
strategy,,,,,,,,,,,,,,,
fixed_0.5,0.063603,0.077139,0.032895,0,0.996634,0.064566,0.083789,0.039474,0,0.997159,-0.000963,-0.006650,-0.006579,0,-0.000525
global_threshold,0.063603,0.975877,0.927632,2,0.154344,0.064566,0.969298,0.907895,2,0.169449,-0.000963,0.006579,0.019737,0,-0.015105
type_specific_thresholds,0.063603,0.940063,0.835526,1,0.270149,0.064566,0.923690,0.789474,1,0.333631,-0.000963,0.016373,0.046052,0,-0.063482


train_rows  train_positive  calibration_rows  \
fold   inspection_type                                                 
fold_1 0                     28277              32              8408   
       1                     22698             269              3868   
       2                     42288             408             16448   
       3                     37264             510             14419   
       4                      1610               4               836   
fold_2 0                     36685              43              6496   
       1                     26566             289              2618   
       2                     58736             500             18964   
       3                     51683             583             15637   
       4                      2446               8               325   
fold_3 0                     43181              93              8985   
       1                     29184             475              5023   
       2                     77700             549              8734   
       3                     67320             622             20747   
       4                      2771              10               698   

                        calibration_positive  evaluation_rows  \
fold   inspection_type                                          
fold_1 0                                  11             6496   
       1                                  20             2618   
       2                                  92            18964   
       3                                  73            15637   
       4                                   4              325   
fold_2 0                                  50             8985   
       1                                 186             5023   
       2                                  49             8734   
       3                                  39            20747   
       4                                   2              698   
fold_3 0                                  14            12107   
       1                                  80             4693   
       2                                  32            14036   
       3                                  23            12673   
       4                                   3              344   

                        evaluation_positive  raw_features  encoded_features  
fold   inspection_type                                                       
fold_1 0                                 50            53                85  
       1                                186            61               111  
       2                                 49            74               119  
       3                                 39            74               112  
       4                                  2            30                52  
fold_2 0                                 14            53                87  
       1                                 80            61               115  
       2                                 32            74               119  
       3                                 23            74               112  
       4                                  3            30                52  
fold_3 0                                  4            53                89  
       1                                 25            61               116  
       2                                  7            74               120  
       3                                  3            74               113  
       4                                  0            30                55

## 8. 최종 타입별 전문가 모델 5개 학습

각 타입에서 전처리기와 delta 기준값은 0~70% Train에만 `fit`합니다.
70~80% Validation에서는 확률과 임계값만 평가하고, 80~100% Test는 손대지 않습니다.


In [9]:
pooled_probability = pd.Series(np.nan, index=validation_df.index, dtype="float64")
models_by_type = {}
preprocessors_by_type = {}
delta_reference_by_type = {}
type_metric_rows = []
training_rows = []

for inspection_type in inspection_types:
    type_train = train_df.loc[train_df[TYPE_COLUMN] == inspection_type].copy()
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type].copy()
    y_train = type_train[TARGET].astype("int8")
    y_validation = type_validation[TARGET].astype("int8")

    assert len(type_train) > 0 and len(type_validation) > 0
    assert y_train.nunique() == 2, f"type={inspection_type} Train에 두 클래스가 없습니다."

    delta_reference = fit_delta_reference(type_train, inspection_type)
    train_model_frame = make_model_frame(type_train, inspection_type, delta_reference)
    validation_model_frame = make_model_frame(type_validation, inspection_type, delta_reference)

    logger.info(
        "model_fit_start type=%d train_rows=%d train_positive=%d valid_rows=%d valid_positive=%d raw_features=%d delta_scale_floor_count=%d",
        inspection_type,
        len(type_train),
        int(y_train.sum()),
        len(type_validation),
        int(y_validation.sum()),
        len(feature_columns_by_type[inspection_type]),
        int((delta_reference['scale'] == 1.0).sum()),
    )

    preprocessor = make_preprocessor(feature_columns_by_type[inspection_type])
    X_train = preprocessor.fit_transform(train_model_frame)
    X_validation = preprocessor.transform(validation_model_frame)
    assert np.isfinite(X_train.data if hasattr(X_train, "data") else X_train).all()
    assert np.isfinite(X_validation.data if hasattr(X_validation, "data") else X_validation).all()

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_train, y_train, verbose=False)
    probability = model.predict_proba(X_validation)[:, 1]
    pooled_probability.loc[type_validation.index] = probability

    metrics = evaluate_probabilities(y_validation, probability)
    metrics["inspection_type"] = inspection_type
    type_metric_rows.append(metrics)
    training_rows.append(
        {
            "inspection_type": inspection_type,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "validation_rows": len(type_validation),
            "validation_positive": int(y_validation.sum()),
            "raw_features": len(feature_columns_by_type[inspection_type]),
            "encoded_features": X_train.shape[1],
            "trees": model.n_estimators,
        }
    )
    models_by_type[inspection_type] = model
    preprocessors_by_type[inspection_type] = preprocessor
    delta_reference_by_type[inspection_type] = delta_reference
    logger.info(
        "model_fit_done type=%d pr_auc=%.6f recall=%.6f fcr=%.6f tp=%d fn=%d",
        inspection_type,
        metrics["pr_auc"],
        metrics["recall"],
        metrics["false_call_reduction"],
        metrics["tp"],
        metrics["fn"],
    )
    del X_train, X_validation, probability
    gc.collect()

assert pooled_probability.notna().all()
pooled_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], pooled_probability),
    name="type_expert_validation",
)
type_metrics = pd.DataFrame(type_metric_rows).set_index("inspection_type")
training_summary = pd.DataFrame(training_rows).set_index("inspection_type")
logger.info("pooled_validation_metrics=%s", pooled_metrics.to_dict())


2026-08-25 16:21:14,077 | INFO | model_fit_start type=0 train_rows=64273 train_positive=111 valid_rows=13289 valid_positive=12 raw_features=53 delta_scale_floor_count=34


2026-08-25 16:21:15,326 | INFO | model_fit_done type=0 pr_auc=0.003705 recall=0.000000 fcr=1.000000 tp=0 fn=12


2026-08-25 16:21:15,420 | INFO | model_fit_start type=1 train_rows=38900 train_positive=580 valid_rows=6422 valid_positive=224 raw_features=61 delta_scale_floor_count=42


2026-08-25 16:21:16,175 | INFO | model_fit_done type=1 pr_auc=0.656618 recall=0.535714 fcr=0.989835 tp=120 fn=104


2026-08-25 16:21:16,366 | INFO | model_fit_start type=2 train_rows=100470 train_positive=588 valid_rows=7161 valid_positive=27 raw_features=74 delta_scale_floor_count=58


...

2026-08-25 16:21:17,818 | INFO | model_fit_start type=3 train_rows=100740 train_positive=648 valid_rows=16252 valid_positive=21 raw_features=74 delta_scale_floor_count=52


2026-08-25 16:21:18,833 | INFO | model_fit_done type=3 pr_auc=0.099674 recall=0.095238 fcr=0.998891 tp=2 fn=19


2026-08-25 16:21:18,869 | INFO | model_fit_start type=4 train_rows=3813 train_positive=13 valid_rows=902 valid_positive=73 raw_features=30 delta_scale_floor_count=11


2026-08-25 16:21:18,932 | INFO | model_fit_done type=4 pr_auc=0.080931 recall=0.000000 fcr=1.000000 tp=0 fn=73


2026-08-25 16:21:18,987 | INFO | pooled_validation_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43586.0, 'fp': 83.0, 'fn': 227.0, 'tp': 130.0, 'accuracy': 0.9929587062190524, 'precision': 0.6103286384976526, 'recall': 0.3641456582633053, 'false_call_reduction': 0.9980993382033021, 'f1': 0.45614035087719296, 'roc_auc': 0.9492585007164606, 'pr_auc': 0.4538261902996763}


## 9. 최종 Validation 결과

아래 `fixed_0.5`는 모델 자체 score 분포 비교용입니다.


In [10]:
count_columns = ["rows", "positive_samples", "tn", "fp", "fn", "tp"]
type_metrics[count_columns] = type_metrics[count_columns].astype("int64")
display(pooled_metrics)
display(
    type_metrics[
        [
            "rows",
            "positive_samples",
            "pr_auc",
            "roc_auc",
            "accuracy",
            "precision",
            "recall",
            "false_call_reduction",
            "f1",
            "tp",
            "fn",
            "fp",
            "tn",
        ]
    ]
)
display(training_summary)


rows                    44026.000000
positive_samples          357.000000
tn                      43586.000000
fp                         83.000000
fn                        227.000000
tp                        130.000000
accuracy                    0.992959
precision                   0.610329
recall                      0.364146
false_call_reduction        0.998099
f1                          0.456140
roc_auc                     0.949259
pr_auc                      0.453826
Name: type_expert_validation, dtype: float64

,rows,positive_samples,pr_auc,roc_auc,accuracy,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
inspection_type,,,,,,,,,,,,,
0,13289,12,0.003705,0.849144,0.999097,0.000000,0.000000,1.000000,0.000000,0,12,0,13277
1,6422,224,0.656618,0.963810,0.973996,0.655738,0.535714,0.989835,0.589681,120,104,63,6135
2,7161,27,0.435472,0.923989,0.997067,0.800000,0.296296,0.999720,0.432432,8,19,2,7132
3,16252,21,0.099674,0.952648,0.997723,0.100000,0.095238,0.998891,0.097561,2,19,18,16213
4,902,73,0.080931,0.500000,0.919069,0.000000,0.000000,1.000000,0.000000,0,73,0,829


,train_rows,train_positive,validation_rows,validation_positive,raw_features,encoded_features,trees
inspection_type,,,,,,,
0,64273,111,13289,12,53,93,400
1,38900,580,6422,224,61,118,400
2,100470,588,7161,27,74,122,400
3,100740,648,16252,21,74,114,400
4,3813,13,902,73,30,58,400


## 10. 최종 Validation에서 공통·타입별 임계값 선택

Recall 99% 이상 후보 중 False Call Reduction이 가장 큰 threshold를 선택합니다.
`004`의 같은 Validation 결과와 차이를 함께 확인합니다.


In [11]:
global_threshold_selection = select_threshold(
    validation_df[TARGET], pooled_probability, min_recall=MIN_RECALL
)

thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = pooled_probability.loc[type_validation.index]
    selection = select_threshold(
        type_validation[TARGET], type_probability, min_recall=MIN_RECALL
    )
    thresholds_by_type[inspection_type] = selection["threshold"]
    selection["inspection_type"] = inspection_type
    type_threshold_rows.append(selection)
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

type_threshold_selection = pd.DataFrame(type_threshold_rows).set_index("inspection_type")
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        pooled_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        pooled_probability,
    ),
    name="type_specific_thresholds",
)

validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": pooled_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T
validation_reference_comparison = (
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
    .join(pd.DataFrame.from_dict(REFERENCE_004_VALIDATION, orient="index"), rsuffix="_ref004")
)
for metric in ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]:
    validation_reference_comparison[f"{metric}_delta_vs_004"] = (
        validation_reference_comparison[metric]
        - validation_reference_comparison[f"{metric}_ref004"]
    )

threshold_summary = pd.concat(
    [
        pd.DataFrame(
            [{"scope": "global", **global_threshold_selection}]
        ).set_index("scope"),
        type_threshold_selection.rename_axis("scope"),
    ],
    axis=0,
)

display(
    threshold_summary[
        ["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]
    ]
)
display(
    validation_strategy_metrics[
        ["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]
    ]
)
display(validation_reference_comparison)
logger.info("global_threshold_selection=%s", global_threshold_selection)
logger.info("type_threshold_selection=%s", type_threshold_selection.to_dict(orient="index"))
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))
logger.info("validation_reference_comparison=%s", validation_reference_comparison.to_dict(orient="index"))


,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000581,357,0.991597,0.640912,354,3,15681,27988
0,0.000188,12,1.000000,0.465768,12,0,7093,6184
1,0.001468,224,0.991071,0.587932,222,2,2554,3644
2,0.000067,27,1.000000,0.148584,27,0,6074,1060
3,0.000581,21,1.000000,0.655166,21,0,5597,10634
4,0.003448,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.453826,0.610329,0.364146,0.998099,0.456140,130.0,227.0,83.0,43586.0
global_threshold,0.453826,0.022077,0.991597,0.640912,0.043192,354.0,3.0,15681.0,27988.0
type_specific_thresholds,0.453826,0.015776,0.994398,0.492844,0.031060,355.0,2.0,22147.0,21522.0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn,pr_auc_ref004,...,tn_ref004,pr_auc_delta_vs_004,precision_delta_vs_004,recall_delta_vs_004,false_call_reduction_delta_vs_004,f1_delta_vs_004,tp_delta_vs_004,fn_delta_vs_004,fp_delta_vs_004,tn_delta_vs_004
fixed_0.5,0.453826,0.610329,0.364146,0.998099,0.456140,130.0,227.0,83.0,43586.0,0.449656,...,43486,0.00417,0.158233,-0.058824,0.002290,0.019093,-21.0,21.0,-100.0,100.0
global_threshold,0.453826,0.022077,0.991597,0.640912,0.043192,354.0,3.0,15681.0,27988.0,0.449656,...,30542,0.00417,-0.004182,0.000000,-0.058485,-0.007972,0.0,0.0,2554.0,-2554.0
type_specific_thresholds,0.453826,0.015776,0.994398,0.492844,0.031060,355.0,2.0,22147.0,21522.0,0.449656,...,24651,0.00417,-0.002548,0.000000,-0.071653,-0.004926,0.0,0.0,3129.0,-3129.0


2026-08-25 16:21:19,187 | INFO | global_threshold_selection={'threshold': 0.0005807904526591301, 'min_recall': 0.99, 'rows': 44026, 'positive_samples': 357, 'tn': 27988, 'fp': 15681, 'fn': 3, 'tp': 354, 'accuracy': 0.6437559623858629, 'precision': 0.022076707202993453, 'recall': 0.9915966386554622, 'false_call_reduction': 0.640912317662415, 'f1': 0.04319180087847731, 'roc_auc': 0.9492585007164606, 'pr_auc': 0.4538261902996763}


2026-08-25 16:21:19,188 | INFO | type_threshold_selection={0: {'threshold': 0.00018842535791918635, 'min_recall': 0.99, 'rows': 13289, 'positive_samples': 12, 'tn': 6184, 'fp': 7093, 'fn': 0, 'tp': 12, 'accuracy': 0.466250282188276, 'precision': 0.001688951442646024, 'recall': 1.0, 'false_call_reduction': 0.46576786924757096, 'f1': 0.0033722073907545315, 'roc_auc': 0.8491438829052748, 'pr_auc': 0.0037050868567874494}, 1: {'threshold': 0.001468099420890212, 'min_recall': 0.99, 'rows': 6422, 'positive_samples': 224, 'tn': 3644, 'fp': 2554, 'fn': 2, 'tp': 222, 'accuracy': 0.601993148551853, 'precision': 0.07997118155619597, 'recall': 0.9910714285714286, 'false_call_reduction': 0.5879315908357534, 'f1': 0.148, 'roc_auc': 0.9638096102429354, 'pr_auc': 0.6566175225364819}, 2: {'threshold': 6.713007314829156e-05, 'min_recall': 0.99, 'rows': 7161, 'positive_samples': 27, 'tn': 1060, 'fp': 6074, 'fn': 0, 'tp': 27, 'accuracy': 0.15179444211702275, 'precision': 0.004425504015735125, 'recall': 1.0

2026-08-25 16:21:19,189 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43586.0, 'fp': 83.0, 'fn': 227.0, 'tp': 130.0, 'accuracy': 0.9929587062190524, 'precision': 0.6103286384976526, 'recall': 0.3641456582633053, 'false_call_reduction': 0.9980993382033021, 'f1': 0.45614035087719296, 'roc_auc': 0.9492585007164606, 'pr_auc': 0.4538261902996763}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 27988.0, 'fp': 15681.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.6437559623858629, 'precision': 0.022076707202993453, 'recall': 0.9915966386554622, 'false_call_reduction': 0.640912317662415, 'f1': 0.04319180087847731, 'roc_auc': 0.9492585007164606, 'pr_auc': 0.4538261902996763}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 21522.0, 'fp': 22147.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.49691091627674555, 'precision': 0.01577637543329482, 'recall': 0.9943977591036415, 'false_call_reduction': 0.4

2026-08-25 16:21:19,189 | INFO | validation_reference_comparison={'fixed_0.5': {'pr_auc': 0.4538261902996763, 'precision': 0.6103286384976526, 'recall': 0.3641456582633053, 'false_call_reduction': 0.9980993382033021, 'f1': 0.45614035087719296, 'tp': 130.0, 'fn': 227.0, 'fp': 83.0, 'tn': 43586.0, 'pr_auc_ref004': 0.4496564930298315, 'precision_ref004': 0.45209580838323354, 'recall_ref004': 0.42296918767507, 'false_call_reduction_ref004': 0.995809384231377, 'f1_ref004': 0.4370477568740955, 'tp_ref004': 151, 'fn_ref004': 206, 'fp_ref004': 183, 'tn_ref004': 43486, 'pr_auc_delta_vs_004': 0.004169697269844785, 'precision_delta_vs_004': 0.15823283011441908, 'recall_delta_vs_004': -0.05882352941176472, 'false_call_reduction_delta_vs_004': 0.002289953971925107, 'f1_delta_vs_004': 0.01909259400309743, 'tp_delta_vs_004': -21.0, 'fn_delta_vs_004': 21.0, 'fp_delta_vs_004': -100.0, 'tn_delta_vs_004': 100.0}, 'global_threshold': {'pr_auc': 0.4538261902996763, 'precision': 0.022076707202993453, 'recal

## 11. 원본 무결성과 종료 확인

원본 데이터와 매핑 파일의 해시는 실행 전후 동일해야 합니다.
Test 구간은 끝까지 hold-out 상태로 유지합니다.


In [12]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "type_models_trained": len(models_by_type),
        "test_predictions_generated": False,
        "fixed_threshold": DECISION_THRESHOLD,
        "global_threshold": global_threshold_selection["threshold"],
        "type_thresholds": thresholds_by_type,
        "log_file": f"docs/peace/{LOG_PATH.name}",
    },
    name="verification",
)
display(verification)
logger.info(
    "source_integrity=PASS test_predictions_generated=False fixed_threshold=%.2f global_threshold=%.8f",
    DECISION_THRESHOLD,
    global_threshold_selection["threshold"],
)
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


dataset_sha256_unchanged                                                   True
mapping_sha256_unchanged                                                   True
type_models_trained                                                           5
test_predictions_generated                                                False
fixed_threshold                                                             0.5
global_threshold                                                       0.000581
type_thresholds               {0: 0.00018842535791918635, 1: 0.0014680994208...
log_file                      docs/peace/0825_peace_016_type_expert_past_del...
Name: verification, dtype: object

2026-08-25 16:21:19,347 | INFO | source_integrity=PASS test_predictions_generated=False fixed_threshold=0.50 global_threshold=0.00058079


2026-08-25 16:21:19,348 | INFO | experiment_complete=0825_peace_016_type_expert_past_change_features


## 12. 결론과 다음 실험

이 실험은 `004` 단일 Walk-forward 타입별 전문가 모델에 "직전 동일 타입 샘플 대비 delta" 집계 피처만 추가한 버전입니다.

해석은 위의 `walk_forward_reference_comparison`, `validation_reference_comparison` 표를 기준으로 합니다.

- `mean_recall_delta_vs_004`, `false_call_reduction_delta_vs_004`가 동시에 양수면 미래 안정성과 오탐 절감이 모두 개선된 것입니다.
- Validation에서만 좋아지고 Walk-forward에서 나빠지면, delta 피처가 최근 구간에만 맞는 과적합 신호일 가능성이 큽니다.
- Walk-forward가 좋아지고 Validation도 유지되면 다음 단계에서 fold ensemble(005 계열)이나 가중치 실험과 결합할 가치가 생깁니다.
